# box-array-to-tensor-with-recipe — faded example 1: Complete the conditional Recipe attachment

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `box-array-to-tensor-with-recipe`. Running the beacon reports progress on the `Backprop: Box array as Tensor + recipe` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Box array as Tensor + recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`box-array-to-tensor-with-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "box-array-to-tensor-with-recipe"
DD_SUBTOPIC = "Backprop: Box array as Tensor + recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Boxing always produces a `MiniTensor`; the `Recipe` is attached **only** when the gate is True. Leaving `recipe = None` on the False branch is what marks a node as a leaf so the reverse pass stops there. The Recipe always carries `(func, raw_args, kwargs, parents)` in that order.

## Faded exercise 1

`box_log` wraps `np.log` over one input. The unbox, forward, gate, and boxing are written. **Complete the step that conditionally attaches the Recipe** so that a Recipe is built iff `requires_grad` is True, leaving `out.recipe = None` otherwise.

**Fill in:** the conditional that, when requires_grad is True, assigns out.recipe = Recipe(np.log, raw_args, kwargs, parents)

In [ ]:
from typing import Callable
from dataclasses import dataclass

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array)
        self.requires_grad = requires_grad
        self.recipe = None

@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict

def box_log(x: MiniTensor, kwargs=None) -> MiniTensor:
    kwargs = kwargs or {}
    raw_args = (x.array,)
    requires_grad = x.requires_grad
    out_raw = np.log(*raw_args, **kwargs)
    parents = {0: x} if requires_grad else {}
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    raise NotImplementedError()  # TODO: when requires_grad is True, assign out.recipe = Recipe(np.log, raw_args, kwargs, parents)
    return out


def _test():
    xg = MiniTensor(np.array([1.0, np.e, np.e ** 2]), requires_grad=True)
    out = box_log(xg)
    assert isinstance(out, MiniTensor)
    assert np.allclose(out.array, np.log(xg.array))
    assert out.requires_grad is True
    assert out.recipe is not None
    assert out.recipe.func is np.log
    assert out.recipe.parents == {0: xg}
    assert out.recipe.parents[0] is xg
    xn = MiniTensor(np.array([1.0, 2.0]), requires_grad=False)
    out2 = box_log(xn)
    assert isinstance(out2, MiniTensor)
    assert np.allclose(out2.array, np.log(xn.array))
    assert out2.requires_grad is False
    assert out2.recipe is None


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from typing import Callable
from dataclasses import dataclass

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array)
        self.requires_grad = requires_grad
        self.recipe = None

@dataclass
class Recipe:
    func: Callable
    args: tuple
    kwargs: dict
    parents: dict

def box_log(x: MiniTensor, kwargs=None) -> MiniTensor:
    kwargs = kwargs or {}
    raw_args = (x.array,)
    requires_grad = x.requires_grad
    out_raw = np.log(*raw_args, **kwargs)
    parents = {0: x} if requires_grad else {}
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(np.log, raw_args, kwargs, parents)
    return out
```
</details>